In [ ]:

# LEGAL CLAUSE CLASSIFICATION - GPT-2 + POST TRAINING QUANTIZATION
# Fine-tuning + Dynamic PTQ + Static PTQ + Performance Dashboard

!pip -q install transformers datasets accelerate

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import copy
import io

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, GPT2ForSequenceClassification
from transformers.pytorch_utils import Conv1D
from torch.ao.quantization import (
    QuantStub, DeQuantStub, prepare, convert,
    get_default_qconfig, quantize_dynamic
)

# --------------------------------------------------
# 1. SETUP
# --------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.manual_seed(42)
np.random.seed(42)

MODEL_NAME = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# --------------------------------------------------
# 2. LOAD LEDGAR DATASET
# --------------------------------------------------

print("\nLoading LEDGAR dataset...")

dataset = load_dataset("coastalcph/lex_glue", "ledgar")

# Select six most frequent labels
label_counts = {}
for label in dataset["train"]["label"]:
    label_counts[label] = label_counts.get(label, 0) + 1

selected_labels = [
    label for label, count in
    sorted(label_counts.items(), key=lambda x: x[1], reverse=True)[:6]
]

label_names = dataset["train"].features["label"].names
label_map = {old: new for new, old in enumerate(selected_labels)}

def prepare_data(split, limit):
    texts, labels = [], []
    for row in dataset[split]:
        if row["label"] in label_map:
            texts.append(row["text"])
            labels.append(label_map[row["label"]])
        if len(texts) >= limit:
            break
    return texts, labels

train_texts, train_labels = prepare_data("train", 500)
test_texts, test_labels = prepare_data("test", 500)

print("Training samples:", len(train_texts))
print("Testing samples:", len(test_texts))
print("Classes:", [label_names[i] for i in selected_labels])

# --------------------------------------------------
# 3. TOKENIZATION
# --------------------------------------------------

class LegalDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            "input_ids": self.encodings["input_ids"][index],
            "attention_mask": self.encodings["attention_mask"][index],
            "labels": self.labels[index]
        }

train_loader = DataLoader(
    LegalDataset(train_texts, train_labels),
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    LegalDataset(test_texts, test_labels),
    batch_size=8
)

# --------------------------------------------------
# 4. FINE-TUNE GPT-2
# --------------------------------------------------

print("\nLoading GPT-2 classifier...")

model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    pad_token_id=tokenizer.pad_token_id
)

model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

print("\nFine-tuning started...")

for epoch in range(2):
    model.train()
    total_loss = 0

    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        "Epoch", epoch + 1,
        "Average Loss:", round(total_loss / len(train_loader), 4)
    )

# --------------------------------------------------
# 5. EVALUATION FUNCTION
# --------------------------------------------------

def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            labels = batch["labels"].to(device)
            inputs = {
                "input_ids": batch["input_ids"].to(device),
                "attention_mask": batch["attention_mask"].to(device)
            }

            outputs = model(**inputs)
            predictions = outputs.logits.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total

original_accuracy = evaluate(model, test_loader, device)
print("\nOriginal GPT-2 Accuracy:", round(original_accuracy, 2), "%")

# --------------------------------------------------
# 6. MODEL SIZE FUNCTION
# --------------------------------------------------

def model_size(model):
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.tell() / (1024 * 1024)

# --------------------------------------------------
# 7. DYNAMIC POST-TRAINING QUANTIZATION
# --------------------------------------------------

print("\nApplying Dynamic Quantization...")

dynamic_model = copy.deepcopy(model).cpu().eval()

# GPT-2 uses Conv1D for transformer projections.
# Convert Conv1D layers into equivalent Linear layers.
def convert_conv1d(module):
    for name, child in list(module.named_children()):
        if isinstance(child, Conv1D):
            linear = nn.Linear(child.weight.shape[0], child.weight.shape[1])
            linear.weight.data.copy_(child.weight.data.T)
            linear.bias.data.copy_(child.bias.data)
            setattr(module, name, linear)
        else:
            convert_conv1d(child)

convert_conv1d(dynamic_model)

dynamic_model = quantize_dynamic(
    dynamic_model,
    {nn.Linear},
    dtype=torch.qint8
)

dynamic_accuracy = evaluate(dynamic_model, test_loader, "cpu")
print("Dynamic Quantized Accuracy:", round(dynamic_accuracy, 2), "%")

# --------------------------------------------------
# 8. STATIC POST-TRAINING QUANTIZATION
# --------------------------------------------------

print("\nApplying Static Quantization to classification head...")

static_model = copy.deepcopy(model).cpu().eval()

# Static quantization wrapper for the final classifier
class StaticQuantLinear(nn.Module):
    def __init__(self, linear):
        super().__init__()
        self.quant = QuantStub()
        self.fc = linear
        self.dequant = DeQuantStub()

    def forward(self, x):
        return self.dequant(self.fc(self.quant(x)))

static_model.score = StaticQuantLinear(static_model.score)

# Configure only the classification head for static quantization
static_model.qconfig = None
static_model.score.qconfig = get_default_qconfig("fbgemm")

prepare(static_model, inplace=True)

# Calibration using representative training samples
static_model.eval()

with torch.no_grad():
    for batch in train_loader:
        static_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

convert(static_model, inplace=True)

static_accuracy = evaluate(static_model, test_loader, "cpu")
print("Static Quantized Accuracy:", round(static_accuracy, 2), "%")

# --------------------------------------------------
# 9. INFERENCE SPEED MEASUREMENT
# --------------------------------------------------

def inference_time(model, loader, device):
    model.eval()
    model.to(device)

    # Warm-up
    with torch.no_grad():
        for batch in list(loader)[:2]:
            model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device)
            )

    total_time = 0
    total_samples = 0

    with torch.no_grad():
        for batch in loader:
            inputs = {
                "input_ids": batch["input_ids"].to(device),
                "attention_mask": batch["attention_mask"].to(device)
            }

            start = time.perf_counter()
            model(**inputs)

            if device == "cuda":
                torch.cuda.synchronize()

            total_time += time.perf_counter() - start
            total_samples += len(batch["labels"])

    return (total_time / total_samples) * 1000

# Benchmark all models on CPU for fair comparison
original_cpu = copy.deepcopy(model).cpu().eval()

original_size = model_size(original_cpu)
dynamic_size = model_size(dynamic_model)
static_size = model_size(static_model)

original_speed = inference_time(original_cpu, test_loader, "cpu")
dynamic_speed = inference_time(dynamic_model, test_loader, "cpu")
static_speed = inference_time(static_model, test_loader, "cpu")

# --------------------------------------------------
# 10. COMPARISON DASHBOARD
# --------------------------------------------------

results = pd.DataFrame({
    "Model": ["Original GPT-2", "Dynamic PTQ", "Static PTQ"],
    "Model Size (MB)": [
        original_size, dynamic_size, static_size
    ],
    "Inference Time (ms/sample)": [
        original_speed, dynamic_speed, static_speed
    ],
    "Accuracy (%)": [
        original_accuracy, dynamic_accuracy, static_accuracy
    ]
})

print("\n========== MODEL PERFORMANCE DASHBOARD ==========")
print(results.round(3).to_string(index=False))

# Model size comparison
plt.figure(figsize=(8, 4))
plt.bar(results["Model"], results["Model Size (MB)"])
plt.title("Model Size Comparison")
plt.ylabel("Size (MB)")
plt.show()

# Inference speed comparison
plt.figure(figsize=(8, 4))
plt.bar(results["Model"], results["Inference Time (ms/sample)"])
plt.title("Inference Time Comparison")
plt.ylabel("Milliseconds per sample (lower is better)")
plt.show()

# Accuracy comparison
plt.figure(figsize=(8, 4))
plt.bar(results["Model"], results["Accuracy (%)"])
plt.title("Legal Clause Tagging Accuracy")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)
plt.show()

# --------------------------------------------------
# 11. SAVE RESULTS
# --------------------------------------------------

torch.save(original_cpu.state_dict(), "original_gpt2.pth")
torch.save(dynamic_model.state_dict(), "dynamic_gpt2.pth")
torch.save(static_model.state_dict(), "static_gpt2.pth")

results.to_csv("quantization_comparison.csv", index=False)

print("\nSaved all three model state dictionaries.")
print("Saved quantization_comparison.csv")
print("\nLEGAL CLAUSE QUANTIZATION PROJECT COMPLETED!")


Device: cpu


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: distilgpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Fine-tuning started...
Epoch 1 Average Loss: 1.9269
